# int8 anchor check (Drive-native)

Verifies the v6b static-int8 candidate against fp32 on the official BMD-45 10k val. T4/CPU ~30-60 min, no training.

Promote gate: int8 mAP50 within -0.02 of fp32 AND int8 fps >= 4.5. Else keep fp32.

## 0. Params — edit once

In [ ]:
VAL_MODE = 'contract'  # 'contract' (JPGs in bmd_contract_live) or 'raw' (PNGs in bmd45_raw)
DRIVE_CONTRACT = '/content/drive/MyDrive/bmd_contract_live'
DRIVE_RAW = '/content/drive/MyDrive/bmd45_raw'
FP32_ONNX = '/content/model.onnx'  # upload: registry fp32 (~12 MB)
INT8_ONNX = '/content/model-v6b-minmax-qdq-fp32tail-200f.onnx'  # upload: candidate (~3 MB)
ROOT = '/tmp/bmd_anchor'
print('init:', VAL_MODE, ROOT)

## 1. Setup

Uploads (file browser): this notebook + `model.onnx` + candidate int8. Val images/labels come from Drive.

In [ ]:
!pip install -q ultralytics onnx onnxruntime-gpu  # gpu build up front: avoids mid-run AutoUpdate + restart
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isfile(FP32_ONNX), 'upload model.onnx first'
assert os.path.isfile(INT8_ONNX), 'upload candidate int8 first'
print('models ok')

## 2. Val pool (Drive-direct, no HF download)

In [ ]:
import json as _json
import cv2
from pathlib import Path
CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle', 'auto']
Path(ROOT, 'images', 'val').mkdir(parents=True, exist_ok=True)
Path(ROOT, 'labels', 'val').mkdir(parents=True, exist_ok=True)
if VAL_MODE == 'contract':
    _imgs = sorted(Path(DRIVE_CONTRACT, 'images', 'val').glob('*.jpg'))
    assert len(_imgs) > 9000, 'contract val incomplete: %d' % len(_imgs)
    import shutil as _sh
    from concurrent.futures import ThreadPoolExecutor as _Pool
    _jobs = []
    for p in _imgs:
        d = Path(ROOT, 'images', 'val', p.name)
        try:
            if d.exists() and d.stat().st_size == p.stat().st_size:
                continue
        except OSError:
            pass
        _jobs.append((p, d))
    _lbs = Path(DRIVE_CONTRACT, 'labels', 'val')
    assert _lbs.is_dir(), 'contract labels/val missing'
    for p in _lbs.glob('*.txt'):
        d = Path(ROOT, 'labels', 'val', p.name)
        try:
            if d.exists() and d.stat().st_size == p.stat().st_size:
                continue
        except OSError:
            pass
        _jobs.append((p, d))
    print('copying %d files (32 threads)...' % len(_jobs))
    def _one(pair):
        s, d = pair
        d.parent.mkdir(parents=True, exist_ok=True)
        _sh.copy(s, d)
        return 1
    _done = 0
    if _jobs:
        with _Pool(max_workers=32) as _ex:
            for _ in _ex.map(_one, _jobs):
                _done += 1
                if _done % 2000 == 0:
                    print('  %d/%d ...' % (_done, len(_jobs)))
    print('pool synced: %d copied' % _done)
else:
    ALIASES = {'motorbike': 'motorcycle', 'moto': 'motorcycle', 'two_wheeler': 'motorcycle', 'two-wheeler': 'motorcycle', 'bicycle': 'bicycle', 'bike': 'bicycle', 'cycle': 'bicycle', 'autorickshaw': 'auto', 'rickshaw': 'auto', 'three_wheeler': 'auto', 'auto_rickshaw': 'auto', 'three-wheeler': 'auto', 'sedan': 'car', 'hatchback': 'car', 'suv': 'car', 'muv': 'car', 'minibus': 'bus', 'mini_bus': 'bus', 'van': 'bus', 'tempo_traveller': 'bus', 'tempo': 'bus', 'lcv': 'truck'}
    def _ci(nm):
        n = (nm or '').lower().strip().replace(' ', '_').replace('-', '_'); m = ALIASES.get(n, n); return CLASSES.index(m) if m in CLASSES else None
    afs = sorted(Path(DRIVE_RAW, 'BMD-45-Val').rglob('*annotations*.json'))
    assert afs, 'no COCO JSON under ' + str(Path(DRIVE_RAW, 'BMD-45-Val'))
    data = _json.loads(afs[0].read_text(encoding='utf-8'))
    cats = {c['id']: c['name'] for c in data.get('categories', [])}
    TABLE = {}
    for a in data.get('annotations', []):
        idx = _ci(cats.get(a.get('category_id')))
        if idx is None: continue
        im = {im['id']: im for im in data.get('images', [])}.get(a.get('image_id'))
        if im is None: continue
        iw, ih = im.get('width') or 1, im.get('height') or 1
        x, y, w, h = a['bbox']
        TABLE.setdefault(Path(str(im.get('file_name', ''))).stem, []).append('%d %.6f %.6f %.6f %.6f' % (idx, (x + w / 2) / iw, (y + h / 2) / ih, w / iw, h / ih))
    n = 0
    for fld in ['images_%03d' % i for i in range(3)]:
        for png in sorted(Path(DRIVE_RAW, 'BMD-45-Val', fld).glob('*.png')):
            if png.stem not in TABLE: continue
            link = Path(ROOT, 'images', 'val', png.stem + '.png')
            if not link.exists():
                try: link.symlink_to(png)
                except OSError: 
                    import shutil as _sh; _sh.copy(png, link)
            open(Path(ROOT, 'labels', 'val', png.stem + '.txt'), 'w').write('\n'.join(TABLE[png.stem]) + '\n')
            n += 1
    print('raw val: %d images linked+labeled' % n)
    assert n > 9000, 'val pool incomplete: %d' % n
lines = ['path: ' + str(Path(ROOT).resolve()), 'train: images/val', 'val: images/val', 'test: images/val', 'names:']
lines += ['  %d: %s' % (i, nm) for i, nm in enumerate(CLASSES)]
open(Path(ROOT) / 'data.yaml', 'w').write('\n'.join(lines) + '\n')
nva = len(list(Path(ROOT, 'images', 'val').glob('*.*')))
print('val pool: %d images' % nva)

## 3. Anchor: fp32 vs int8 mAP50 + fps (the promotion gate)

In [ ]:
import time as _t
import onnxruntime as _ort
from ultralytics import YOLO
import numpy as _np, cv2 as _cv
DATA = '/tmp/bmd_anchor/data.yaml'
m_fp = YOLO(FP32_ONNX).val(data=DATA, verbose=False, device=0)  # T4 CUDA; falls back to CPU with warning
map_fp = round(float(m_fp.box.map50), 4)
print('fp32 mAP50: %.4f' % map_fp)
m_q = YOLO(INT8_ONNX).val(data=DATA, verbose=False, device=0)
map_q = round(float(m_q.box.map50), 4)
print('int8 mAP50: %.4f (delta %+.4f)' % (map_q, map_q - map_fp))
_fps = {}
# CPU fps is intentional: the promotion gate (>=4.5) is a low-tier CPU number
for tag, mp in (('fp32', FP32_ONNX), ('int8', INT8_ONNX)):
    s = _ort.InferenceSession(mp, providers=['CPUExecutionProvider'])
    im = _cv.resize(_cv.imread(str(sorted((__import__('pathlib').Path(ROOT, 'images', 'val')).glob('*.*'))[0])), (640, 640))
    rgb = _cv.cvtColor(im, _cv.COLOR_BGR2RGB).astype(_np.float32) / 255.0
    feed = {s.get_inputs()[0].name: _np.transpose(rgb, (2, 0, 1))[_np.newaxis]}
    t0 = _t.perf_counter()
    for _ in range(50): s.run(None, feed)
    _fps[tag] = round(50 / (_t.perf_counter() - t0), 2)
print('fps:', _fps)
ok = (map_q >= map_fp - 0.02) and (_fps['int8'] >= 4.5)
print('VERDICT:', 'PROMOTE — flip prefer_int8:true + copy to registry' if ok else 'KEEP fp32 — do NOT promote')
open('/tmp/bmd_anchor/anchor_report.txt', 'w').write('fp32 %.4f int8 %.4f delta %+.4f fps %s verdict %s\n' % (map_fp, map_q, map_q - map_fp, _fps, 'PROMOTE' if ok else 'KEEP'))
from google.colab import files
files.download('/tmp/bmd_anchor/anchor_report.txt')
print('DONE — paste anchor_report.txt back here.')